In [15]:
import os.path
import gc
import pandas as pd
import pickle as pkl
import numpy as np
import random
from tqdm import tqdm
import json
from sklearn import metrics
from utils import get_texts_for_ptype
from sklearn import metrics
from scipy.optimize import minimize_scalar
from custom_models.custom_model import BatchVectorizerXGBoost
from ray import tune

In [16]:
neg_to_pos_ratio = 1

inputs = "titles_abstracts"

filename_pos_human = "../data/pubmed_articles_pos_human.csv"
filename_neg_human = "../data/pubmed_articles_neg_human.csv"

with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
    prop_typ_id_dict = pkl.load(f)

ptypes = prop_typ_id_dict.keys()

In [17]:
def predict(df, db=0):
    return [int(x>=db) for x in df ]

In [4]:
def get_scores(predictions):
    return [np.exp(p[1])/(np.exp(p[0]) + np.exp(p[1])) for p in predictions]

In [5]:
neg_pos_ratio_dict = {
    "is_rct": 5585042/489295,
    "is_sr": 5659292/121063,
    "is_casecontrol": 5628219/65790,
    "is_cohort": 5632460/78688,
    "is_guideline": 5676411/33826
}

def calc_precision(recall, specificity, neg_pos_ratio):
    return recall / (recall + neg_pos_ratio * (1 - specificity) + 1e-15)

def calc_f1(recall, precision):
    return 2*recall*precision/(recall+precision + 1e-15)

In [6]:
def find_db(db):
    y_pred_test = predict(df, db)
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    return abs(recall_test - 0.985)

In [7]:
results_file = "results_titles_abstracts_gbdt_features1000.json"
    
with open(results_file, "r") as f:
    results_list = json.load(f)

In [8]:
gbdt_metrics_adjusted_results_path = "results_adjusted_metrics_gbdt.json"

In [12]:
metrics_lst = []
for results_json in tqdm(results_list):
    ptype = results_json["ptype"]
    X_train, X_val, X_test, y_train, y_val, y_test = get_texts_for_ptype(ptype, inputs="titles_abstracts", neg_to_pos_ratio=1, reset_index=True)
    
    bvc = BatchVectorizerXGBoost(
            objective="binary:logistic",
            # max_depth=config["max_depth"],
            max_depth=18,
            gamma=results_json["gamma"],
            reg_alpha=results_json["reg_alpha"],
            reg_lambda=results_json["reg_lambda"],
            colsample_bytree=results_json["colsample_bytree"],
            min_child_weight=results_json["min_child_weight"],
            # n_estimators=config["n_estimators"],
            n_estimators=180,
            seed=0,
        )
    
    print("Fitting begins")
    bvc.fit(X_train, y_train)
    predictions = bvc.predict_proba(X_test)
    
    scores = get_scores(predictions)
        
    def find_db(db):
        y_pred_test = predict(scores, db)
        recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
        return abs(recall_test - 0.985)

    auc_roc = metrics.roc_auc_score(y_test, scores)

    res = minimize_scalar(find_db)
    db = res["x"]
    y_pred_test = predict(scores, db)
    
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
    precision_test = calc_precision(recall_test, spec_test, neg_pos_ratio_dict[ptype])
    f1_test = calc_f1(recall_test, precision_test)
    
    metrics_dict = {
        "ptype": ptype,
        "y_test": y_test,
        "logits": predictions,
        "db": db,
        "scores": scores,
        "auc_roc": auc_roc,
        "recall_test": recall_test,
        "spec_test": spec_test,
        "precision_test": precision_test,
        "f1_test": f1_test
    }
    metrics_lst.append(metrics_dict)
    
    if os.path.exists(gbdt_metrics_adjusted_results_path):
        with open(gbdt_metrics_adjusted_results_path, "rb") as f:
            metrics_lst = pkl.load(f)
    
    metrics_lst.append(metrics_dict)
    with open(gbdt_metrics_adjusted_results_path, "wb") as f:
        pkl.dump(metrics_lst, f)
        
    del bvc
    gc.collect()

  0%|                                                     | 0/5 [00:00<?, ?it/s]
0it [00:00, ?it/s]
1it [00:01,  1.33s/it]
2it [00:02,  1.36s/it]
3it [00:03,  1.30s/it]
4it [00:05,  1.29s/it]
5it [00:06,  1.32s/it]
6it [00:07,  1.32s/it]
7it [00:09,  1.34s/it]
8it [00:10,  1.32s/it]
9it [00:11,  1.32s/it]
10it [00:13,  1.34s/it]
11it [00:14,  1.30s/it]
12it [00:15,  1.29s/it]
13it [00:17,  1.32s/it]
14it [00:18,  1.33s/it]
15it [00:19,  1.36s/it]
16it [00:21,  1.37s/it]
17it [00:22,  1.33s/it]
18it [00:23,  1.34s/it]
19it [00:25,  1.37s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:

20it [00:26,  1.37s/it]
21it [00:28,  1.34s/it]
22it [00:29,  1.36s/it]
23it [00:30,  1.35s/it]
24it [00:31,  1.31s/it]
25it [00:33,  1.31s/it]
26it [00:34,  1.33s/it]
27it [00:35,  1.33s/it]
28it [00:37,  1.31s/it]
29it [00:38,  1.29s/it]
30it [00:39,  1.30s/i

Fitting begins


/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[14:06:45] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


 20%|████████▊                                   | 1/5 [03:30<14:00, 210.14s/it]
0it [00:00, ?it/s]
1it [00:01,  1.34s/it]
2it [00:02,  1.37s/it]
3it [00:03,  1.32s/it]
4it [00:05,  1.31s/it]
5it [00:06,  1.34s/it]
6it [00:08,  1.34s/it]
7it [00:09,  1.36s/it]
8it [00:10,  1.34s/it]
9it [00:12,  1.34s/it]
10it [00:13,  1.36s/it]
11it [00:14,  1.32s/it]
12it [00:15,  1.31s/it]
13it [00:17,  1.33s/it]
14it [00:18,  1.35s/it]
15it [00:20,  1.38s/it]
16it [00:21,  1.39s/it]
17it [00:22,  1.35s/it]
18it [00:24,  1.35s/it]
19it [00:25,  1.39s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:

20it [00:27,  1.39s/it]
21it [00:28,  1.37s/it]
22it [00:29,  1.39s/it]
23it [00:31,  1.38s/it]
24it [00:32,  1.35s/it]
25it [00:33,  1.35s/it]
26it [00:35,  1.37s/it]
27it [00:36,  1.37s/it]
28it [00:37,  1.35s/it]
29it [00:39,  1.32s/it]
30it [00:40,  1.34s/i

...

 80%|███████████████████████████████████▏        | 4/5 [14:39<03:42, 222.58s/it]
0it [00:00, ?it/s]
1it [00:01,  1.33s/it]
2it [00:02,  1.36s/it]
3it [00:03,  1.30s/it]
4it [00:05,  1.29s/it]
5it [00:06,  1.33s/it]
6it [00:07,  1.32s/it]
7it [00:09,  1.34s/it]
8it [00:10,  1.32s/it]
9it [00:11,  1.32s/it]
10it [00:13,  1.34s/it]
11it [00:14,  1.30s/it]
12it [00:15,  1.30s/it]
13it [00:17,  1.32s/it]
14it [00:18,  1.33s/it]
15it [00:19,  1.36s/it]
16it [00:21,  1.37s/it]
17it [00:22,  1.33s/it]
18it [00:23,  1.33s/it]
19it [00:25,  1.37s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:

20it [00:26,  1.37s/it]
21it [00:28,  1.35s/it]
22it [00:29,  1.37s/it]
23it [00:30,  1.36s/it]
24it [00:32,  1.33s/it]
25it [00:33,  1.33s/it]
26it [00:34,  1.35s/it]
27it [00:36,  1.35s/it]
28it [00:37,  1.33s/it]
29it [00:38,  1.31s/it]
30it [00:39,  1.32s/i

Fitting begins


/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[14:21:25] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


100%|████████████████████████████████████████████| 5/5 [18:06<00:00, 217.39s/it]
